In [ ]:
!git clone https://github.com/Exploration-Lab/CS780-OBELIX
%cd CS780-OBELIX

Cloning into 'CS780-OBELIX'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 9), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 993.81 KiB | 4.82 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/CS780-OBELIX/CS780-OBELIX


In [ ]:
!pip install -r requirements.txt
!pip install stable-baselines3
!pip install gymnasium

In [ ]:
import gymnasium as gym
from gymnasium import spaces

import numpy as np
import random

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

from obelix import OBELIX

In [ ]:
ACTIONS = ["L45","L22","FW","R22","R45"]

In [ ]:
class ObelixEnv(gym.Env):

    def __init__(self):
        super().__init__()

        self.env = None

        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(18,),
            dtype=np.float32
        )

        self.action_space = spaces.Discrete(5)

    def reset(self, seed=None, options=None):

        difficulty = np.random.choice([0,2,3])

        self.env = OBELIX(
            scaling_factor=2,
            difficulty=difficulty,
            max_steps=500   # faster training
        )

        obs = self.env.reset()

        return obs.astype(np.float32), {}

    def step(self, action):

        action_str = ["L45","L22","FW","R22","R45"][action]

        obs, reward, done = self.env.step(action_str, render=False)

        return obs.astype(np.float32), reward, done, False, {}

In [ ]:
def make_env():
    return Monitor(ObelixEnv(), filename=None)

In [ ]:
env = DummyVecEnv([make_env for _ in range(2)])

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(18,128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,5)
        )

    def forward(self,x):
        return self.net(x)


class SaveModelCallback(BaseCallback):

    def __init__(self, save_freq=50000, verbose=1):
        super().__init__(verbose)
        self.save_freq = save_freq

    def _on_step(self):

        if self.num_timesteps % self.save_freq == 0:

            print("Saving model at step:", self.num_timesteps)

            policy_net = self.model.policy.mlp_extractor.policy_net
            action_net = self.model.policy.action_net

            submission_model = Net()

            # ✅ COPY CORRECTLY (128 → 128)
            submission_model.net[0].weight.data = policy_net[0].weight.data
            submission_model.net[0].bias.data = policy_net[0].bias.data

            submission_model.net[2].weight.data = policy_net[2].weight.data
            submission_model.net[2].bias.data = policy_net[2].bias.data

            submission_model.net[4].weight.data = action_net.weight.data
            submission_model.net[4].bias.data = action_net.bias.data

            torch.save(submission_model.state_dict(), "weights.pth")

            # also save PPO model
            self.model.save("ppo_full")

        return True

In [ ]:
import os
from stable_baselines3 import PPO

MODEL_PATH = "ppo_full"

if os.path.exists(MODEL_PATH + ".zip"):
    print("Loading existing model...")

    model = PPO.load(MODEL_PATH, env=env)

else:
    print("Creating new model...")

    policy_kwargs = dict(net_arch=[128,128])

    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,
        n_steps=2048,
        batch_size=256,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.02,
        policy_kwargs=policy_kwargs
    )

Creating new model...
Using cpu device


In [ ]:
callback = SaveModelCallback(save_freq=30000)

In [ ]:
model.learn(
    total_timesteps=100000,
    callback=callback
)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 500       |
|    ep_rew_mean     | -2.16e+03 |
| time/              |           |
|    fps             | 87        |
|    iterations      | 1         |
|    time_elapsed    | 46        |
|    total_timesteps | 4096      |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 500           |
|    ep_rew_mean          | -6.65e+03     |
| time/                   |               |
|    fps                  | 87            |
|    iterations           | 2             |
|    time_elapsed         | 94            |
|    total_timesteps      | 8192          |
| train/                  |               |
|    approx_kl            | 6.0644234e-06 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.61         |
|    explained_variance   | 0.0013

In [ ]:
model.learn(
    total_timesteps=1000000,
    callback=callback,
    reset_num_timesteps=False
)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 498       |
|    ep_rew_mean     | -4.96e+03 |
| time/              |           |
|    fps             | 88        |
|    iterations      | 1         |
|    time_elapsed    | 46        |
|    total_timesteps | 287230    |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 496           |
|    ep_rew_mean          | -5.26e+03     |
| time/                   |               |
|    fps                  | 88            |
|    iterations           | 2             |
|    time_elapsed         | 92            |
|    total_timesteps      | 291326        |
| train/                  |               |
|    approx_kl            | 1.2070086e-06 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.61         |
|    explained_variance   | 0.0002

KeyboardInterrupt: 

In [ ]:
model.save("ppo_full")

In [ ]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(18,128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,5)
        )

    def forward(self,x):
        return self.net(x)

submission_model = Net()

In [ ]:
policy_net = model.policy.mlp_extractor.policy_net
action_net = model.policy.action_net

In [ ]:
submission_model.net[0].weight.data = policy_net[0].weight.data
submission_model.net[0].bias.data   = policy_net[0].bias.data

submission_model.net[2].weight.data = policy_net[2].weight.data
submission_model.net[2].bias.data   = policy_net[2].bias.data

submission_model.net[4].weight.data = action_net.weight.data
submission_model.net[4].bias.data   = action_net.bias.data

In [ ]:
torch.save(submission_model.state_dict(), "weights.pth")

In [ ]:
from google.colab import files
files.download("weights.pth")
files.download("ppo_full.zip")